In [3]:
from google.colab import userdata
api_key = userdata.get("GEMINI_API_KEY")

## Setup

In [4]:
!pip install -q langchain langchain-google-genai pydantic pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 18.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


In [24]:
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
import pandas as pd
import time
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=userdata.get("GEMINI_API_KEY"),
    max_retries=2,
)

## Requirement 1 — Prompt Template & first call

In [25]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a creative social-media writer based in Cairo who writes for local cafes and shops."),
    ("human", "Write an Instagram post draft for this product:\n{description}"),
])

raw_chain = prompt | llm

example = "New iced karak chai with cardamom, served cold in a tall glass"
print(raw_chain.invoke({"description": example}).text)

☕❄️ **Beat the Cairo heat, but make it *desi*!** 

Let’s be real—we love our classic Karak, but when the afternoon sun is hitting different in Zamalek or Maadi, a piping hot cup is the last thing on your mind. 

Enter our **NEW Iced Karak Chai** 🧊👑

We took that rich, spiced, slow-brewed black tea and condensed milk goodness you’re addicted to, gave it a heavy dash of aromatic cardamom, and poured it over a tall glass of crushed ice. It’s creamy, it’s spicy, it’s aggressively refreshing, and it’s officially your new summer obsession. 

Come through, grab a tall glass, and let’s pretend Cairo traffic doesn’t exist for a solid ten minutes. 😌✨

📍 Drop by [Insert Cafe Name], [Insert Neighborhood]
🛵 Or order via DM / [Insert delivery link] for your desk-side rescue!

#CairoCafes #IcedKarak #ChaiLover #CairoEats #KarakChai #CoffeeVibesCairo #SummerInCairo #GoodVibesOnly


## Requirement 2 — Structured output with Pydantic

In [26]:
class SocialPost(BaseModel):
    title: str = Field(description="Punchy post title, max 60 characters")
    caption: str = Field(description="Engaging caption, max 280 characters")
    hashtags: list[str] = Field(description="3 to 5 hashtags, each starting with #", min_length=3, max_length=5)
    alt_text: str = Field(description="Short accessibility description of the image")

structured_llm = llm.with_structured_output(SocialPost)
structured_chain = prompt | structured_llm

descriptions = [
    "New iced karak chai with cardamom, served cold in a tall glass",
    "Fresh-baked pistachio basbousa with rose syrup, made every morning",
    "Handmade dark brown leather laptop sleeve that fits 13-inch laptops",
]

for d in descriptions:
    post = structured_chain.invoke({"description": d})
    print(type(post), isinstance(post, SocialPost))
    print(post, "\n")
    time.sleep(13)

<class '__main__.SocialPost'> True
title='Beat the Cairo Heat with Iced Karak!' caption='Your favorite spiced chai, now served freezing cold! Our new Iced Karak Chai is infused with aromatic cardamom and poured over ice in a tall refreshing glass. Come grab yours today and stay cool in the city! 🧊☕️' hashtags=['#IcedKarak', '#CairoCafes', '#CardamomChai', '#CoffeeShopCairo'] alt_text='A tall glass filled with iced karak chai and cardamom, condensation on the glass against a warm Cairo cafe background' 

<class '__main__.SocialPost'> True
title='Morning sweetness, Cairo style!' caption='Wake up to our freshly baked pistachio basbousa drenched in fragrant rose syrup. Baked from scratch every single morning just the way you love it. Drop by for your morning slice and a cup of qahwa!' hashtags=['#Basbousa', '#CairoEats', '#PistachioLovers', '#CairoCafes'] alt_text='A golden slice of pistachio basbousa garnished with crushed nuts and rose petals on a ceramic plate.' 

<class '__main__.Socia

## Requirement 3 — Sequential chain: Arabic translation

In [27]:
translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a professional English-to-Arabic translator for social media. Keep the tone friendly and natural. Return only the translation."),
    ("human", "Translate this caption into Arabic:\n{caption}"),
])

from langchain_core.output_parsers import StrOutputParser

translate_chain = translate_prompt | llm | StrOutputParser()

def generate_post(description: str) -> dict:
    post = structured_chain.invoke({"description": description})
    time.sleep(13)
    arabic = translate_chain.invoke({"caption": post.caption}).strip()
    time.sleep(13)
    return {**post.model_dump(), "arabic_caption": arabic}

In [28]:
rows = []
for d in descriptions:
    result = generate_post(d)
    rows.append({"description": d, **result})

df = pd.DataFrame(rows, columns=["description", "title", "caption", "hashtags", "alt_text", "arabic_caption"])
df["hashtags"] = df["hashtags"].apply(lambda h: " ".join(h))
df.to_csv("posts.csv", index=False, encoding="utf-8-sig")
df

,description,title,caption,hashtags,alt_text,arabic_caption
0,"New iced karak chai with cardamom, served cold...",Beat the Cairo heat with Iced Karak!,Your favorite spiced brew just got a frosty ma...,#IcedKarak #CairoCafes #CardamomChai #SummerDr...,A tall glass filled with creamy iced karak cha...,مشروبكم المفضل المُتبل صار بحلة صيفية منعشة! ن...
1,Fresh-baked pistachio basbousa with rose syrup...,Morning Pistachio Basbousa,Sweeten your Cairo morning with our freshly ba...,#CairoCafes #Basbousa #PistachioLove #SweetCairo,A close-up shot of a golden-brown pistachio ba...,حلّي صباحك في القاهرة مع بسبوسة الفستق الطازجة...
2,Handmade dark brown leather laptop sleeve that...,Upgrade Your Carry,Give your 13-inch laptop the home it deserves....,#CairoCrafts #LeatherGoods #LaptopSleeve #Made...,A handmade dark brown leather laptop sleeve de...,دلل لاب توبك مقاس 13 بوصة بالجراب الذي يستحقه....
